# AI Lab: Ross-Macdonald Sensitivity Indices and Intervention Ranking

**Book companion exercise**

**Considerations exercised:** 11 (correct sensitivity), 12 (basic vs effective $\mathcal{R}$)
**Estimated runtime:** ~20 minutes
**Companion audit:** [`14_vector_intervention_ranking.md`](../ai-audit/14_vector_intervention_ranking.md)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/exercises/ai-lab/14_vector_intervention_lab.ipynb)


## What this lab does

Computes closed-form normalized sensitivity indices for the Ross-Macdonald $\mathcal{R}_0$, verifies them numerically, and uses them to rank candidate interventions categorically (biting > vector reduction > recovery acceleration). Builds the intuition for why bednets are the highest-priority lever in vector-borne disease control.

## Setup

In [ ]:
import sys, os
# AI Lab notebooks live in exercises/ai-lab/ and need access to shared/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'python')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import math
from shared import book_style, BOOK_COLORS
from shared.parameters import baseline_chapter_07
book_style()
rng = np.random.default_rng(42)


## Step 1 — Ross-Macdonald $\mathcal{R}_0$ and closed-form sensitivities

$$\mathcal{R}_0 = \sqrt{\frac{a^2 b_h b_v m}{\mu_v \gamma_h}}$$
where $a$ = biting rate, $b_h, b_v$ = transmission probabilities per bite, $m$ = vector-to-host ratio, $\mu_v$ = vector mortality, $\gamma_h$ = host recovery.

In [ ]:
import math
# Baseline parameters (illustrative dengue-like values)
params = dict(a=0.4, b_h=0.4, b_v=0.4, m=5.0, mu_v=1/14, gamma_h=1/7)

def R0(p):
    return math.sqrt(p['a']**2 * p['b_h'] * p['b_v'] * p['m']
                     / (p['mu_v'] * p['gamma_h']))

R0_base = R0(params)
print(f'Baseline R_0 = {R0_base:.3f}')

# Closed-form normalized sensitivities (analytical):
# d log R_0 / d log a    = +1
# d log R_0 / d log m    = +1/2
# d log R_0 / d log b_h  = +1/2
# d log R_0 / d log b_v  = +1/2
# d log R_0 / d log mu_v = -1/2
# d log R_0 / d log g_h  = -1/2
closed = dict(a=+1.0, m=+0.5, b_h=+0.5, b_v=+0.5, mu_v=-0.5, gamma_h=-0.5)
for p, s in closed.items():
    print(f'  Closed-form S^R0_{p} = {s:+.2f}')


## Step 2 — Verify numerically

In [ ]:
h = 1e-4
for p in closed:
    perturbed = dict(params)
    perturbed[p] = params[p] * (1 + h)
    R0_p = R0(perturbed)
    numeric = (math.log(R0_p) - math.log(R0_base)) / (math.log(perturbed[p]) - math.log(params[p]))
    match = abs(numeric - closed[p]) < 1e-3
    print(f'  {p:>8s}: numeric S = {numeric:+.4f}, closed = {closed[p]:+.2f}  match = {match}')
    assert match, f'Sensitivity mismatch for {p}'
print('\nVerified: numerical sensitivities match closed-form expressions.')


## Step 3 — Intervention ranking

Compare three interventions, each implementing a 50% relative change in one parameter:

In [ ]:
interventions = [
    ('Biting-rate reduction (bednets)', 'a', 0.5),
    ('Vector reduction (insecticide)', 'm', 0.5),
    ('Faster recovery (treatment)', 'gamma_h', 1.5),  # 50% increase reduces R_0
    ('Vector mortality (larvicide)', 'mu_v', 1.5),    # 50% increase reduces R_0
]

print(f"{'Intervention':<40s} {'param':<10s} {'new R_0':>9s} {'reduction':>10s}")
for name, p, factor in interventions:
    perturbed = dict(params)
    perturbed[p] = params[p] * factor
    R0_new = R0(perturbed)
    reduction = (R0_base - R0_new) / R0_base * 100
    print(f'{name:<40s} {p:<10s} {R0_new:>9.3f} {reduction:>9.1f}%')

print()
print('Biting-rate reduction is twice as effective per unit relative change')
print('because S^R0_a = 1 while the other parameters have |S| = 0.5.')


## Step 4 — Tornado plot

In [ ]:
labels = list(closed.keys())
vals = [closed[k] for k in labels]
order = sorted(range(len(labels)), key=lambda i: -abs(vals[i]))
labels_s = [labels[i] for i in order]
vals_s = [vals[i] for i in order]
colors = [BOOK_COLORS['secondary'] if v > 0 else BOOK_COLORS['primary'] for v in vals_s]

fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(labels_s))
ax.barh(y, vals_s, color=colors, edgecolor='black')
ax.set_yticks(y); ax.set_yticklabels(labels_s)
ax.invert_yaxis()
ax.axvline(0, color='black', lw=0.6)
ax.set_xlabel(r'normalized sensitivity index $S^{\mathcal{R}_0}_p$')
ax.set_title('Ross-Macdonald: closed-form sensitivities')
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
plt.show()


## What's next

- Open Project `vector_control_in_real_settings.md` extends to spatially heterogeneous transmission
- Companion audit: `../ai-audit/14_vector_intervention_ranking.md`
- Book Chapter 11 figures `figs/ch14_biting_decomposition.pdf` visualizes the biting-rate dominance